<a href="https://colab.research.google.com/github/larpita/Stock-Recommendation-System/blob/main/Stock_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Stock Recommendation System**

A decision-support system that suggests Buy / Hold / Sell based on fundamental indicators + historical performance.

In [2]:
# installing yahoo finance for historical price data and gradio
!pip install yfinance
!pip install gradio

In [3]:
#importing required libararies
import yfinance as yf
import pandas as pd #to handle data
import numpy as np #for calculations
import gradio as gr


In [9]:
def analyze_stock(ticker):
    ticker = ticker.upper()
    if "." not in ticker: #handling Indian stocks(NSE)
         ticker = ticker + ".NS"

    stock = yf.Ticker(ticker) #taking stock entered by user
    price_data = stock.history(period="5y")#fetch last 5 yrs price data
    if price_data.empty: #if there is no history then the stock isn't available or there is no data of it
        return "Invalid ticker or no data available."
    start_price = price_data['Close'].iloc[0]
    end_price = price_data['Close'].iloc[-1]
    # calculating 5-year cagr of that stock
    cagr = (end_price / start_price) ** (1/5) - 1
    #getting fundamental data of the stock
    info = stock.info
    pe_ratio = info.get("trailingPE")
    roe = info.get("returnOnEquity")
    debt_equity = info.get("debtToEquity")
    profit_margin = info.get("profitMargins")
    #scoring the stock based on rules
    #initialization
    score = 0
    reasons = []
    #applying fundamental rules
    #P/E ratio
    if pe_ratio and pe_ratio < 25:
        score += 1
        reasons.append("Reasonable P/E ratio")
    #ROE
    if roe and roe > 0.15:
        score += 1
        reasons.append("Strong Return on Equity")
    #Debt
    if debt_equity and debt_equity < 1:
        score += 1
        reasons.append("Low debt level")
    #Profit margin
    if profit_margin and profit_margin > 0.1:
        score += 1
        reasons.append("Healthy profit margins")
    #Growth
    if cagr > 0.08:
        score += 1
        reasons.append("Strong growth potential")
    #Recommendation
    if score >= 4:
        recommendation = "BUY"
    elif score >= 2:
        recommendation = "HOLD"
    else:
        recommendation = "SELL"
    #output
    result = f"""
Stock: {ticker}
Recommendation: {recommendation}
Score: {score} / 5

Reasons:
"""
    for r in reasons:
        result += f"- {r}\n"

    result += f"\n5-Year CAGR: {round(cagr*100, 2)}%"

    return result
#UI
ui = gr.Interface(
    fn=analyze_stock,
    inputs=gr.Textbox(
        lines=1,
        placeholder="Enter stock ticker (e.g. AAPL, TCS.NS)",
        label="Stock Ticker"
    ),
    outputs=gr.Textbox(
        lines=15,
        label="Stock Analysis Output"
    ),
    title="Fundamental-Based Stock Recommendation System",
    description="Enter a stock ticker to get Buy / Hold / Sell recommendation based on fundamentals and 5-year performance."
)

ui.launch()












It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://02a071384cd02433f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
